# **Load Parameters**

In [0]:
import traceback

try:
    catalog = dbutils.widgets.get("catalog")
    bronze_schema = dbutils.widgets.get("bronze_schema")
    silver_schema = dbutils.widgets.get("silver_schema")
    bronze_table = dbutils.widgets.get("bronze_table")
    silver_table = dbutils.widgets.get("silver_table")
except Exception as e:
    print("Error getting notebook parameters")
    print(traceback.format_exc())
    raise e

## **Create Silver Table**

In [0]:
try:
    create_table_query = f"""
    CREATE TABLE IF NOT EXISTS {catalog}.{silver_schema}.{silver_table} (
      invoice_line_no STRING,
      date DATE,
      store_number INT,
      store_name STRING,
      address STRING,
      city STRING,
      zip_code STRING,
      county_number INT,
      county STRING,
      category_code INT,
      category_name STRING,
      vendor_number INT,
      vendor_name STRING,
      item_number INT,
      item_description STRING,
      pack INT,
      bottle_volume_ml INT,
      state_bottle_cost DECIMAL(10, 2),
      state_bottle_retail DECIMAL(10, 2),
      bottles_sold INT,
      sale_dollars DECIMAL(10, 2),
      volume_sold_liters DECIMAL(10, 3),
      volume_sold_gallons DECIMAL(10, 3),
      saved_date TIMESTAMP
    )
    USING DELTA
    """
    
    print(f"Creating or verifying table {catalog}.{silver_schema}.{silver_table}...")
    spark.sql(create_table_query)

except Exception as e:
    print(f"Error creating Silver table: {catalog}.{silver_schema}.{silver_table}")
    print(traceback.format_exc())
    raise e

## **Load, Cleanse, and Transform Data**

In [0]:
from pyspark.sql.functions import col, upper, trim, to_date
from pyspark.sql.types import IntegerType, DecimalType, DateType
import traceback

try:
    print(f"Loading data from {catalog}.{bronze_schema}.{bronze_table}...")
    df_sales_raw = spark.table(f"{catalog}.{bronze_schema}.{bronze_table}")
    
    initial_row_count = df_sales_raw.count()
    print(f"Initial row count from Bronze: {initial_row_count}")
    
    # Define critical columns for the 'na.drop'
    critical_cols = [
        'invoice_line_no', 'date', "store", "county_number", "category",
        "vendor_no", "itemno", "pack", "bottle_volume_ml", "state_bottle_cost",
        "state_bottle_retail", "sale_bottles"
    ]
    
    print("Applying cleansing, type casting, and business rule transformations...")
    
    df_silver = (df_sales_raw
        
        # 1. Filter corrupt rows (from Bronze layer)
        .filter(col("is_corrupted") == False)
        
        # 2. Filter rows with nulls in critical business columns
        .na.drop(subset=critical_cols)
        
        # 3. Fill nulls for non-critical string attributes
        .na.fill({
            "address": "UNKNOWN ADDRESS",
            "city": "UNKNOWN CITY",
            "zipcode": "00000", # Use string for zip code
            "county": "UNKNOWN COUNTY",
            "category_name": "UNCATEGORIZED",
            "vendor_name": "UNKNOWN VENDOR",
            "im_desc": "NO DESCRIPTION"
        })
        
        # 4. Cast columns to their correct data types
        .withColumn("date", col("date").cast(DateType()))
        .withColumn("store_number", col("store").cast(IntegerType()))
        .withColumn("county_number", col("county_number").cast(IntegerType()))
        .withColumn("category_code", col("category").cast(IntegerType()))
        .withColumn("vendor_number", col("vendor_no").cast(IntegerType()))
        .withColumn("item_number", col("itemno").cast(IntegerType()))
        .withColumn("pack", col("pack").cast(DecimalType(10, 2)).cast(IntegerType()))
        .withColumn("bottle_volume_ml", col("bottle_volume_ml").cast(DecimalType(10, 2)).cast(IntegerType()))
        .withColumn("bottles_sold", col("sale_bottles").cast(IntegerType()))
        .withColumn("state_bottle_cost", col("state_bottle_cost").cast(DecimalType(10, 2)))
        .withColumn("state_bottle_retail", col("state_bottle_retail").cast(DecimalType(10, 2)))
        
        # 5. Filter out rows with zero values in key measures
        .filter(
            (col("state_bottle_cost") > 0) & 
            (col("state_bottle_retail") > 0) &
            (col('bottle_volume_ml') > 0)
        )
        
        # 6. Recalculate key metrics for data integrity
        .withColumn("sale_dollars", (col("state_bottle_retail") * col("bottles_sold")).cast(DecimalType(10, 2)))
        .withColumn("volume_sold_liters", ((col("bottle_volume_ml") * col("bottles_sold")) / 1000).cast(DecimalType(10, 3)))
        .withColumn("volume_sold_gallons", ((col("bottle_volume_ml") * col("bottles_sold")) / 3785.411784).cast(DecimalType(10, 3)))
        
        # 7. Select and rename columns for the final Silver schema
        .select(
            col("invoice_line_no").alias("invoice_line_no"), # Renamed in your file but schema is same
            col("date"),
            col("store_number"),
            col("name").alias("store_name"),
            col("address"),
            col("city"),
            col("zipcode").alias("zip_code"),
            col("county_number"),
            col("county"),
            col("category_code"),
            col("category_name"),
            col("vendor_number"),
            col("vendor_name"),
            col("item_number"),
            col("im_desc").alias("item_description"),
            col("pack"),
            col("bottle_volume_ml"),
            col("state_bottle_cost"),
            col("state_bottle_retail"),
            col("bottles_sold"),
            col("sale_dollars"),
            col("volume_sold_liters"),
            col("volume_sold_gallons"),
            col("saved_date")
        )
    )
    
    
    # Log final counts
    final_row_count = df_silver.count()
    print(f"Final row count for Silver: {final_row_count}")
    print(f"Total rows removed by quality checks: {initial_row_count - final_row_count}")

except Exception as e:
    print("Error during Silver transformation")
    print(traceback.format_exc())
    raise e

## **Load Data into Silver Table**

In [0]:
try:
    print(f"Writing data to {catalog}.{silver_schema}.{silver_table}...")
    
    # Create a temporary view from the final DataFrame
    df_silver.createOrReplaceTempView("tmp_sales_final")

    # Use INSERT OVERWRITE for an atomic and idempotent write
    insert_query = f"""
    INSERT OVERWRITE {catalog}.{silver_schema}.{silver_table}
    SELECT * FROM tmp_sales_final
    """
    
    spark.sql(insert_query)
    print("Write to Silver table complete.")
    
except Exception as e:
    print("Error writing to Silver table")
    print(traceback.format_exc())
    raise e
# finally:
#     # Unpersist the DataFrame to free up memory
#     df_silver.unpersist()

# Final exit to signal success
dbutils.notebook.exit("Silver layer processing complete.")